# Fine-tuning an LLM on PubMedQA using GRPO

In [12]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

In [ ]:
from unsloth import is_bfloat16_supported
max_seq_length = 2048 
lora_rank = 64

QWEN = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = QWEN,
    max_seq_length = max_seq_length,
    load_in_4bit = True, 
    fast_inference = True, 
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, 
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], 
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", 
    random_state = 1234,
)

## Data Processing

In [ ]:
from datasets import load_dataset

ds = load_dataset("pragmaticcs/pubmedqa-prompts")

df_l = ds["labeled"].to_pandas()
df_a = ds["artificial_balanced"].select(range(1600)).to_pandas()

In [16]:
SYSTEM_PROMPT = """\
You are a biomedical research assistant skilled in explaining your reasoning. Your task is to answer questions based on the provided scientific abstract.First, provide the long answer - a detailed, step by step explanation of your reasoning with evidence from the abstract. Then, give your final answer in the following format:
<long_answer>
[long answer, citing specific evidence from the abstract]
</long_answer>
<answer>
[yes | no | maybe]
</answer>
"""

### Creating the train/test datasets

In [17]:
def create_conversation(system, question, answer=None):
    conversation = [
      {"role": "system", "content": system},
      {"role": "user", "content": question},
    ]
    if answer != None:
      conversation.append({"role": "assistant", "content": answer})
    return conversation

def create_dataset(df) -> Dataset:
    data = []
    for id, row in df.iterrows():
      question, answer, long_answer = row['prompt'], row['answer'], row['long_answer']
      convo = create_conversation(SYSTEM_PROMPT, question)
      data.append({
          'prompt': convo,
          'answer': answer,
          'long_answer': long_answer
      })
    return Dataset.from_list(data)


In [18]:
def create_train_test_split(df, ratio: float):
    total_rows = len(df)
    train_rows = int(total_rows * ratio)
    return (df[:train_rows], df[train_rows:])

In [19]:
train_df, test_df = create_train_test_split(df_l, ratio=0.8)
train_df = pd.concat([df_a, train_df], ignore_index=True)

In [20]:
train_dataset, test_dataset = create_dataset(train_df), create_dataset(test_df)

## Training Preparation

### Helper libraries and models

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Reward Functions

In [21]:
def extract_xml_answer(text: str) -> str:
    """Helper function"""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_xml_reasoning(text: str) -> str:
    """Helper function"""
    reasoning = text.split("<long_answer>")[-1]
    reasoning = reasoning.split("</long_answer>")[0]
    return reasoning.strip()

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<long_answer>\n.*?\n</long_answer>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [1.0 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    """Helper function for xmlcount_reward_function"""
    count = 0.0
    if text.count("<long_answer>\n") == 1:
        count += 0.25
    if text.count("\n</long_answer>\n") == 1:
        count += 0.25
    if text.count("\n<answer>\n") == 1:
        count += 0.25
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.25
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that encourages all 4 reasoning tokens in the response."""
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

def length_penalty(reasoning, long_answer):
    """Helper function for length_penalty_reward_func"""
    target_length = len(word_tokenize(reasoning.lower()))
    actual_length =  len(word_tokenize(long_answer.lower()))
    ratio = min(actual_length, target_length) / max(actual_length, target_length) if max(actual_length, target_length) > 0 else 0.0
    return ratio

def correct_reasoning_length_reward_func(completions, long_answer, **kwargs) -> list[float]:
    """Reward function that encourages similiar length of reasoning to long_answer from the dataset."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_reasonings = [extract_xml_reasoning(r) for r in responses]
    rewards = [length_penalty(r, l) for r, l in zip(extracted_reasonings, long_answer)]
    return rewards

def keyword_overlap_reward_func(completions, long_answer, **kwargs) -> list[float]:
    """Reward function based on keyword overlap between reasoning and long_answer."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_reasonings = [extract_xml_reasoning(r) for r in responses]
    rewards = []
    for _, (r, l) in enumerate(zip(extracted_reasonings, long_answer)):
        if not r or not l:
            rewards.append(0.0)
            continue
        reasoning_tokens = word_tokenize(r.lower())
        long_answer_tokens = word_tokenize(l.lower())

        reasoning_keywords = [lemmatizer.lemmatize(w) for w in reasoning_tokens if not w in stop_words and w.isalnum()]
        long_answer_keywords = [lemmatizer.lemmatize(w) for w in long_answer_tokens if not w in stop_words and w.isalnum()]
        if not reasoning_keywords or not long_answer_keywords:
            rewards.append(0.0)
            continue
        common_keywords = set(reasoning_keywords) & set(long_answer_keywords)
        denominator = min(len(set(reasoning_keywords).union(set(long_answer_keywords))), len(long_answer_keywords))
        overlap_ratio = len(common_keywords) / denominator if denominator > 0 else 0.0
        reward = max(0.0, overlap_ratio)
        rewards.append(reward)
    return rewards

def paragraph_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that encourages proper paragraph formatting while penalizing bullet points and lists."""
    responses = [completion[0]["content"] for completion in completions]
    extracted_reasonings = [extract_xml_reasoning(r) for r in responses]
    rewards = []
    for reasoning in extracted_reasonings:
        if not reasoning:
            rewards.append(0.0)
            continue
        bullet_pattern = r'^\s*[\*\-•]\s+' 
        numbered_pattern = r'^\s*\d+[\.)]\s+'
        lines = reasoning.split('\n')
        bullet_lines = sum(1 for line in lines if re.match(bullet_pattern, line))
        numbered_lines = sum(1 for line in lines if re.match(numbered_pattern, line))
        total_lines = len(lines)
        if total_lines == 0:
            rewards.append(0.0)
            continue
        list_penalty = (bullet_lines + numbered_lines) / total_lines
        sentences = re.split(r'[.!?]+', reasoning)
        sentences = [s.strip() for s in sentences if s.strip()]
        sentence_line_ratio = len(sentences) / max(total_lines, 1)
        paragraph_score = min(sentence_line_ratio, 3.0) / 3.0
        newline_density = reasoning.count('\n') / max(len(reasoning), 1)
        excess_newline_penalty = min(newline_density * 5, 1.0)
        reward = 0.7 * (1.0 - list_penalty) + 0.3 * paragraph_score - 0.5 * excess_newline_penalty
        rewards.append(max(0.0, min(reward, 1.0)))
    return rewards

def no_answer_in_reasoning_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that penalizes the direct use of 'yes', 'no', or 'maybe' in the reasoning."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_reasonings = [extract_xml_reasoning(r) for r in responses]
    penalty_words = ['yes', 'no', 'maybe']
    rewards = []
    for reasoning in extracted_reasonings:
        if not reasoning:
            rewards.append(0.0) 
            continue
        reasoning_lower = reasoning.lower()
        reward = 1.0
        for word in penalty_words:
            if word in reasoning_lower:
                reward = 0.0
        rewards.append(reward)
    return rewards


def answer_correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function based on correctness of the extracted answer."""
    responses = [completion[0]['content'] for completion in completions]
    extracted_answers = [extract_xml_answer(r) for r in responses]
    answer_rewards = [1.0 if r == a else -1.0 for r, a in zip(extracted_answers, answer)]
    print('#'*20, 
          f"Question:\n{prompts[0][-1]['content']}", 
          f"\nAnswer:\n{answer[0]}", 
          f"\nResponses:\n{"\n".join([f'[Start]\n{response}[End]\n' for response in responses])}")
    return answer_rewards

## Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, 
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 6,
    gradient_accumulation_steps = 1, 
    num_generations = 6, 
    max_prompt_length = 4096,
    max_completion_length = 2048,
    num_train_epochs = 1, 
    max_grad_norm = 0.1,
    report_to = "none", 
    output_dir = "outputs",
    save_strategy = "steps",
    save_steps = 500,
    reward_weights=[
        0.25, 
        0.15,
        0.15,

        0.05,
        0.15,
        0.15,
        0.10,
    ] # Sum of rewards = 1.0
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        answer_correctness_reward_func,
        keyword_overlap_reward_func,
        correct_reasoning_length_reward_func,
        
        no_answer_in_reasoning_reward_func,
        xmlcount_reward_func,
        strict_format_reward_func,
        paragraph_format_reward_func,
    ],
    args = training_args,
    train_dataset = train_dataset,
)
trainer.train(resume_from_checkpoint = False)

## Evaluation

In [23]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1,
    top_p = 0.95,
    max_tokens = 1024,
    seed=1234
)

correct = 0
for index, row in test_df.iterrows():
    question = row['prompt']
    prompt = tokenizer.apply_chat_template([
        {"role" : "system", "content" : SYSTEM_PROMPT},
        {"role" : "user", "content" : question},
    ], tokenize = False, add_generation_prompt = True)

    long_answer = row['long_answer']
    true_answer = row['answer']

    output = model.fast_generate(
        prompt,
        sampling_params = sampling_params,
        lora_request = model.load_lora("grpo_saved_lora"),
    )[0].outputs[0].text

    pred_answer = extract_xml_answer(output).strip().lower()
    if pred_answer == true_answer:
        correct += 1
    
    print(f"--- Example {index+1} ---")
    print(f"Question: \n{question}")
    print(f"Expected Answer: '{true_answer}'")
    print(f"Expected Long Answer: '{long_answer}'")
    print(f"\nResponse: \n{output}\n")
    print(f"Correct: {pred_answer == true_answer}")
    print("-" * 20)
    
print(f"Accuracy: {correct/len(test_df)*100:.2f}%")

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it, est. speed input: 406.94 toks/s, output: 76.36 toks/s]


--- Example 801 ---
Question: 
BACKGROUND: Skin diseases are the most frequently recognized occupational diseases in Denmark. The prognosis for occupational contact dermatitis is often poor.
OBJECTIVES: To investigate the prognosis, assessed by eczema, job status and skin-related quality of life, among patients allergic to rubber chemicals and latex (ubiquitous allergens) and epoxy (nonubiquitous allergen), 2 years after recognition of occupational allergic contact dermatitis.
METHODS: From a cohort of all patients recognized as having occupational dermatitis by the Danish National Board of Industrial Injuries in 2010, 199 patients with relevant rubber allergy (contact allergy to rubber chemicals or contact urticaria from latex) or epoxy allergy were identified. Follow-up consisted of a questionnaire covering current severity of eczema, employment, exposure and quality of life.
RESULTS: The response rate was 75%. Clearance of eczema was reported by 11% of patients and 67% reported impr

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s, est. speed input: 513.34 toks/s, output: 75.88 toks/s]


--- Example 802 ---
Question: 
OBJECTIVE: Assessing the clinical course of inflammatory bowel disease (IBD) patients consists of periodical clinical evaluations and laboratory tests. We aimed to assess the role of calprotectin tests in predicting clinical relapse in IBD patients.
METHODS: Ninety-seven patients with ulcerative colitis (UC) and 65 with Crohn's disease (CD) in clinical remission were prospectively included in the study. A 10-g stool sample was collected for calprotectin assay. The cutoff level was set at 130 mg/kg of feces. Patients were followed up for 1 yr after the test or until relapse. The cumulative proportion of relapses was estimated by the Kaplan-Meier analysis. Statistics for equality of survival distribution were tested using the log-rank test.
RESULTS: The calprotectin test was positive in 44 UC patients and 26 of them relapsed within a year, while 11 of 53 UC patients with a negative calprotectin test relapsed within the same time frame. Thirty CD patients ha

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s, est. speed input: 608.19 toks/s, output: 72.87 toks/s]


...

--- Example 998 ---
Question: 
BACKGROUND: We analyzed the pharmacokinetic-pharmacodynamic relationship of vancomycin to determine the drug exposure parameters that correlate with the efficacy and nephrotoxicity of vancomycin in patients with methicillin-resistant Staphylococcus aureus pneumonia and evaluated the need to use peak concentration in therapeutic drug monitoring (TDM).
METHODS: Serum drug concentrations of 31 hospitalized patients treated with vancomycin for methicillin-resistant S. aureus pneumonia were collected.
RESULTS: Significant differences in trough concentration (Cmin)/minimum inhibitory concentration (MIC) and area under the serum concentration-time curve (AUC0-24)/MIC were observed between the response and non-response groups. Significant differences in Cmin and AUC0-24 were observed between the nephrotoxicity and non-nephrotoxicity groups. Receiver operating characteristic curves revealed high predictive values of Cmin/MIC and AUC0-24/MIC for efficacy and of Cmi

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s, est. speed input: 535.95 toks/s, output: 79.16 toks/s]


--- Example 999 ---
Question: 
PURPOSE: This investigation assesses the effect of platelet-rich plasma (PRP) gel on postoperative pain, swelling, and trismus as well as healing and bone regeneration potential on mandibular third molar extraction sockets.
PATIENTS AND METHODS: A prospective randomized comparative clinical study was undertaken over a 2-year period. Patients requiring surgical extraction of a single impacted third molar and who fell within the inclusion criteria and indicated willingness to return for recall visits were recruited. The predictor variable was application of PRP gel to the socket of the third molar in the test group, whereas the control group had no PRP. The outcome variables were pain, swelling, and maximum mouth opening, which were measured using a 10-point visual analog scale, tape, and millimeter caliper, respectively. Socket healing was assessed radiographically by allocating scores for lamina dura, overall density, and trabecular pattern. Quantitative 

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it, est. speed input: 474.53 toks/s, output: 81.91 toks/s]

--- Example 1000 ---
Question: 
OBJECTIVE: The reduced use of sugars-containing (SC) liquid medicines has increased the use of other dose forms, potentially resulting in more widespread dental effects, including tooth wear. The aim of this study was to assess the erosive potential of 97 paediatric medicines in vitro.
METHODS: The study took the form of in vitro measurement of endogenous pH and titratable acidity (mmol). Endogenous pH was measured using a pH meter, followed by titration to pH 7.0 with 0.1-M NaOH.
RESULTS: Overall, 55 (57%) formulations had an endogenous pH of<5.5. The mean (+/- SD) endogenous pH and titratable acidity for 41 SC formulations were 5.26 +/- 1.30 and 0.139 +/- 0.133 mmol, respectively; for 56 sugars-free (SF) formulations, these figures were 5.73 +/- 1.53 and 0.413 +/- 1.50 mmol (P>0.05). Compared with their SC bioequivalents, eight SF medicines showed no significant differences for pH or titratable acidity, while 15 higher-strength medicines showed lower p